# Lesson 22 - NumPy Arrays

**Goals:** Create and inspect arrays; select, reshape, broadcast, and summarize numerical readings.

**Before you begin:** Lessons 4, 9, 12, and 14 cover lists, copying, transformations, and iteration. Previous stage: [21_mysql_with_python.ipynb](21_mysql_with_python.ipynb).
This lesson defines all its own inputs and runs independently on Python 3.11+.

**Setup:** `requirements.txt` (NumPy). See the README shared requirements instructions. Install from a terminal only when you choose;
these notebooks never install packages. No core example needs a network, API key, database server, or download.
All records are fictional.

Run cells top to bottom with **Shift+Enter**. Predict the output before running each example,
change one input, rerun the relevant cells, and explain what changed. Restart the kernel and
Run All after restoring the original inputs. Practice cells intentionally contain no answers.
Attempt the exercises and project before opening `instructor_solutions/22_numpy_arrays_solutions.ipynb`.


## 1. From Python records to numerical arrays

Lists can hold mixed objects. A NumPy array stores a rectangular block with one data type
(`dtype`), which makes numerical operations compact and usually faster for sizable numerical data.
Vectorization asks NumPy to perform a whole-array operation in compiled routines; it does not
guarantee every tiny calculation is faster. Keep dictionaries for named records and arrays for measurements.

`ndim` counts axes, `shape` gives lengths along them, and `size` counts all elements.
Rows below are sites; columns are three daily kWh readings. Predict the shape before running.


In [ ]:
import numpy as np

readings = np.array([[10, 12, 14], [20, 22, 24]], dtype=np.float64)
print(readings)
print(readings.ndim, readings.shape, readings.size, readings.dtype)
assert readings.shape == (2, 3)
assert readings.dtype == np.float64


## 2. Create arrays and choose a dtype

`zeros` and `ones` create initialized arrays. `arange` resembles range with an excluded stop;
`linspace` requests a fixed count and includes both endpoints by default. Prefer linspace to
floating steps when endpoint/count matters. A seeded generator provides repeatable synthetic data.
Integer arrays cannot preserve fractional assignments. `astype` explicitly converts the dtype.


In [ ]:
print(np.zeros((2, 2)))
print(np.ones(3, dtype=np.int64))
print(np.arange(0, 6, 2))
print(np.linspace(0, 1, 5))
rng = np.random.default_rng(22)
print(rng.integers(1, 5, size=4))
counts = np.array([1, 2], dtype=np.int64)
counts[0] = 1.9  # Fraction is truncated; this is not rounding.
print(counts, counts.astype(float))
assert counts.tolist() == [1, 2]


## 3. Index, slice, and copy deliberately

`array[row, column]` accesses one value. Slices exclude their stop as in lists, but basic
NumPy slices usually share the original storage: changing a slice changes the source. Use `.copy()`
for an independent result. Boolean/advanced indexing instead creates a copy.
Predict whether the following original changes. Change the copy and compare.


In [ ]:
sample = np.array([[10., 12., 14.], [20., 22., 24.]])
print(sample[1, 2], sample[:, 1], sample[0, :2])
view = sample[:, :2]
view[0, 0] = 99
independent = sample.copy()
independent[0, 0] = -1
print(sample[0, 0], independent[0, 0])
assert sample[0, 0] == 99


## 4. Boolean masks select matching measurements

A comparison creates one boolean per element. Indexing with that mask returns matching values.
Combine array conditions with `&`, `|`, and `~`, wrapping each comparison in parentheses.
Python `and` expects a single truth value and cannot combine whole masks.
`np.where(condition, yes, no)` chooses element by element without changing the source.


In [ ]:
mask = (readings >= 12) & (readings < 23)
selected = readings[mask]
print(mask, selected, sep="\n")
print(np.where(readings > 20, "review", "normal"))
assert selected.tolist() == [12., 14., 20., 22.]


## 5. Reshape and understand axes

Reshaping rearranges the interpretation of existing values while keeping the element count.
One `-1` lets NumPy infer a dimension. A reshape may share memory; use copy when independence matters.
For a `(sites, days)` table, `axis=0` collapses sites and leaves one value per day;
`axis=1` collapses days and leaves one per site. Transpose (`.T`) exchanges the two axes.


In [ ]:
six = np.arange(6)
print(six.reshape(2, 3))
print(readings.T.shape)
print("Per day:", readings.sum(axis=0))
print("Per site:", readings.sum(axis=1))
print("Mean/min/max:", readings.mean(), readings.min(), readings.max())
assert readings.sum(axis=0).tolist() == [30, 34, 38]
assert readings.sum(axis=1).tolist() == [36, 66]


## 6. Vectorization and broadcasting

For lists, `* 2` repeats items; for arrays it multiplies values. `*` is elementwise;
matrix multiplication uses `@`. Broadcasting aligns shapes from the right: paired dimensions
must be equal or one of them must be 1. A scalar broadcasts everywhere; a `(3,)` tariff aligns
with three columns. Per-site factors need shape `(2, 1)`, not `(2,)`.
Predict the first site's bill; change only the final day's tariff and explain which totals move.


In [ ]:
print([10, 12] * 2, np.array([10, 12]) * 2)
tariff = np.array([2., 3., 4.])  # Currency units per kWh, one per day.
cost = readings * tariff
print(cost, cost.sum(axis=1), sep="\n")
site_factors = np.array([1., 0.5]).reshape(-1, 1)
print(readings * site_factors)
assert cost.sum(axis=1).tolist() == [112, 202]


## 7. Conversion boundaries and debugging

Mixed text/numbers can turn the entire array into strings. Ragged nested lists do not form
a numerical rectangle. Diagnose errors by printing shape and dtype before doing arithmetic.
`tolist()` produces ordinary nested Python values for JSON. Check finite values with `isfinite`;
NaN is not equal to itself. Empty arrays need an explicit aggregation policy.
Energy examples use floats; retain Lesson 12's Decimal policy for exact accounting.


In [ ]:
mixed = np.array([10, "12"])
print(mixed.dtype, mixed.astype(float) + 1)
for operation in [lambda: np.array([[1, 2], [3]]),
                  lambda: np.arange(5).reshape(2, 3),
                  lambda: readings + np.ones(2)]:
    try:
        operation()
    except ValueError:
        print("Rejected: inspect rectangularity, element count, or broadcast shapes.")
dirty = np.array([10., np.nan, np.inf])
print(dirty[np.isfinite(dirty)])
assert readings.tolist() == [[10., 12., 14.], [20., 22., 24.]]


## Practice lab

Use the supplied inputs, add your own comments, and compare with the check values. Use `practice_`
names where helpful. Increase the difficulty one exercise at a time. Checks describe results;
they are not complete implementations. Test a boundary or malformed input when applicable.


### Exercise 1. Inspect an array

Create a float array from `[5, 10, 15]`; report shape, dtype, and total.

**Check:** `(3,)`, float64, total 30.


In [ ]:
practice_values = [5, 10, 15]
# TODO: convert, inspect, and summarize.


### Exercise 2. Select a column

From `[[2,4,6],[8,10,12]]`, select the middle column and last row.

**Check:** `[4,10]` and `[8,10,12]`.


In [ ]:
practice_grid = np.array([[2, 4, 6], [8, 10, 12]])
# TODO: select without a loop.


### Exercise 3. Filter invalid readings

Keep finite, nonnegative values from `[10,-1,nan,20,inf,0]`.

**Check:** `[10,20,0]`, mean 10.


In [ ]:
practice_dirty = np.array([10., -1., np.nan, 20., np.inf, 0.])
# TODO: combine masks; do not remove valid zeros.


### Exercise 4. Broadcast a tariff

Multiply `[[1,2,3],[4,5,6]]` by daily tariffs `[2,3,4]`; total each row.

**Check:** `[20,47]`.


In [ ]:
practice_kwh = np.array([[1,2,3],[4,5,6]])
practice_tariffs = np.array([2,3,4])
# TODO: elementwise cost and row totals.


### Exercise 5. Normalize each site without mutating input

Write `center_rows(values)` for nonempty finite 2D arrays: subtract each row mean, using `keepdims=True` to preserve a column dimension. Reject invalid dimensions or nonfinite data.

**Check:** `[[1,3],[10,14]]` becomes `[[-1,1],[-2,2]]`; original unchanged.


In [ ]:
def practice_center_rows(values):
    # TODO: validate shape and finiteness; return a new centered array.
    pass


## Chapter project

Build `energy_report(values, tariffs)` for a nonempty finite nonnegative 2D kWh table and a
finite nonnegative 1D tariff of matching column length. Return JSON-ready `site_kwh`, `site_cost`,
and `grand_cost`. Validate shapes before multiplying; never mutate inputs.

**Check:** `[[10,12,14],[20,22,24]]` and `[2,3,4]` give site kWh `[36,66]`,
cost `[112,202]`, grand cost 314. Reject a short tariff and a negative reading.


In [ ]:
def energy_report(values, tariffs):
    # TODO: validate, calculate, and return ordinary Python values.
    pass


## Reflection and knowledge check

1. How does list multiplication differ from array multiplication?
2. Which axis leaves one total per site?
3. Why can a slice change its source?
4. When is broadcasting valid, and how can it silently represent the wrong meaning?

Choose one changed input, predict the new result, and explain your observation to a colleague.


## Official references

- [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)
- [Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
